# Week 8 — UI Layer & Response Generation
### Real-Time Urban Data Integration Platform
**Team 4** | Kritika Gupta | Rohit Manohar Saggam | Nandini Devi Poreddy

**Modules covered in this notebook:**
1. Natural Language Response Generator
2. Display Components
3. Edge Case Testing
4. End-to-End Flow Integration
5. Interactive CLI Interface

## Step 1: Install Required Libraries

In [ ]:
!pip install requests -q
print('Libraries installed successfully.')

## Step 2: Reuse Week 7 Functions
Copy the core functions from Week 7 to use in this notebook.

In [ ]:
import requests
import json
from datetime import datetime

# ---- Location Retrieval (from Week 7) ----
def get_coordinates(city: str) -> dict:
    try:
        url = 'https://geocoding-api.open-meteo.com/v1/search'
        params = {'name': city, 'count': 1, 'language': 'en', 'format': 'json'}
        response = requests.get(url, params=params, timeout=10)
        data = response.json()
        if 'results' not in data or len(data['results']) == 0:
            return {'error': f"City '{city}' not found."}
        loc = data['results'][0]
        return {
            'city': loc['name'], 'country': loc.get('country', ''),
            'latitude': loc['latitude'], 'longitude': loc['longitude'],
            'timezone': loc.get('timezone', 'auto')
        }
    except Exception as e:
        return {'error': str(e)}


# ---- Weather Agent (from Week 7) ----
def get_weather(city: str) -> dict:
    coords = get_coordinates(city)
    if 'error' in coords:
        return coords
    weather_descriptions = {
        0: 'Clear sky', 1: 'Mainly clear', 2: 'Partly cloudy', 3: 'Overcast',
        45: 'Foggy', 48: 'Icy fog', 51: 'Light drizzle', 53: 'Moderate drizzle',
        61: 'Slight rain', 63: 'Moderate rain', 65: 'Heavy rain',
        71: 'Slight snow', 73: 'Moderate snow', 75: 'Heavy snow',
        80: 'Slight showers', 81: 'Moderate showers', 82: 'Violent showers',
        95: 'Thunderstorm'
    }
    try:
        url = 'https://api.open-meteo.com/v1/forecast'
        params = {
            'latitude': coords['latitude'], 'longitude': coords['longitude'],
            'current_weather': True,
            'hourly': 'temperature_2m,relativehumidity_2m,apparent_temperature',
            'forecast_days': 1
        }
        response = requests.get(url, params=params, timeout=10)
        data = response.json()
        if 'current_weather' not in data:
            return {'error': 'Weather data unavailable.'}
        current = data['current_weather']
        hourly = data.get('hourly', {})
        return {
            'city': coords['city'], 'country': coords['country'],
            'temperature_celsius': current['temperature'],
            'temperature_fahrenheit': round(current['temperature'] * 9/5 + 32, 1),
            'feels_like_celsius': hourly.get('apparent_temperature', [None])[0],
            'humidity_percent': hourly.get('relativehumidity_2m', [None])[0],
            'wind_speed_kmh': current['windspeed'],
            'weather_condition': weather_descriptions.get(current['weathercode'], 'Unknown'),
            'hourly_temps': hourly.get('temperature_2m', []),
            'hourly_humidity': hourly.get('relativehumidity_2m', []),
            'data_source': 'Open-Meteo API'
        }
    except Exception as e:
        return {'error': str(e)}


# ---- AQI Agent (from Week 7) ----
def get_aqi_category(pm25: float) -> dict:
    if pm25 <= 12:
        return {'category': 'Good', 'color': 'Green', 'recommendation': 'Safe for all activities.'}
    elif pm25 <= 35.4:
        return {'category': 'Moderate', 'color': 'Yellow', 'recommendation': 'Sensitive groups should take care.'}
    elif pm25 <= 55.4:
        return {'category': 'Unhealthy for Sensitive Groups', 'color': 'Orange', 'recommendation': 'Limit prolonged outdoor activity.'}
    elif pm25 <= 150.4:
        return {'category': 'Unhealthy', 'color': 'Red', 'recommendation': 'Reduce outdoor exertion.'}
    else:
        return {'category': 'Very Unhealthy', 'color': 'Purple', 'recommendation': 'Avoid outdoor activities.'}


def get_aqi(city: str) -> dict:
    coords = get_coordinates(city)
    if 'error' in coords:
        return coords
    try:
        url = 'https://air-quality-api.open-meteo.com/v1/air-quality'
        params = {
            'latitude': coords['latitude'], 'longitude': coords['longitude'],
            'current': 'pm10,pm2_5,carbon_monoxide,nitrogen_dioxide,ozone',
            'hourly': 'pm2_5,pm10',
            'timezone': 'auto'
        }
        response = requests.get(url, params=params, timeout=10)
        data = response.json()
        if 'current' not in data:
            return {'error': 'AQI data unavailable.'}
        current = data['current']
        pm25 = current.get('pm2_5', 0) or 0
        aqi_info = get_aqi_category(pm25)
        hourly = data.get('hourly', {})
        return {
            'city': coords['city'], 'country': coords['country'],
            'pm2_5_ugm3': pm25,
            'pm10_ugm3': current.get('pm10', 0),
            'nitrogen_dioxide_ugm3': current.get('nitrogen_dioxide', 0),
            'ozone_ugm3': current.get('ozone', 0),
            'aqi_category': aqi_info['category'],
            'aqi_color': aqi_info['color'],
            'health_recommendation': aqi_info['recommendation'],
            'hourly_pm25': hourly.get('pm2_5', []),
            'data_source': 'Open-Meteo Air Quality API'
        }
    except Exception as e:
        return {'error': str(e)}


print('Week 7 functions loaded successfully.')

## Step 6: Natural Language Response Generator
**Owner: Kritika**

Converts raw numerical data into human-friendly explanations.
Adds contextual insights, trend summaries, and warnings.

In [ ]:
def generate_full_response(city: str, intent: str = 'combined') -> str:
    """
    Generates a complete natural language response for a city query.

    Args:
        city (str): City name
        intent (str): 'weather_only', 'aqi_only', or 'combined'

    Returns:
        str: Formatted natural language report
    """
    lines = []
    lines.append(f'Urban Environmental Report — {city}')
    lines.append('=' * 50)

    if intent in ['weather_only', 'combined']:
        weather = get_weather(city)
        if 'error' not in weather:
            temp_c = weather['temperature_celsius']
            temp_f = weather['temperature_fahrenheit']
            feels = weather.get('feels_like_celsius', 'N/A')
            humidity = weather.get('humidity_percent', 'N/A')
            wind = weather['wind_speed_kmh']
            condition = weather['weather_condition']

            lines.append(f'\nWeather Conditions')
            lines.append('-' * 30)
            lines.append(f'Temperature  : {temp_c}°C / {temp_f}°F')
            lines.append(f'Feels Like   : {feels}°C')
            lines.append(f'Humidity     : {humidity}%')
            lines.append(f'Wind Speed   : {wind} km/h')
            lines.append(f'Conditions   : {condition}')
            lines.append(f'\nInsight: {generate_weather_insight(weather)}')
        else:
            lines.append(f'Weather: Data unavailable — {weather["error"]}')

    if intent in ['aqi_only', 'combined']:
        aqi = get_aqi(city)
        if 'error' not in aqi:
            lines.append(f'\nAir Quality')
            lines.append('-' * 30)
            lines.append(f'PM2.5        : {aqi["pm2_5_ugm3"]} µg/m³')
            lines.append(f'PM10         : {aqi["pm10_ugm3"]} µg/m³')
            lines.append(f'NO2          : {aqi["nitrogen_dioxide_ugm3"]} µg/m³')
            lines.append(f'AQI Category : {aqi["aqi_category"]} ({aqi["aqi_color"]})')
            lines.append(f'\nInsight: {generate_aqi_insight(aqi)}')
        else:
            lines.append(f'Air Quality: Data unavailable — {aqi["error"]}')

    lines.append('\n' + '=' * 50)
    lines.append(f'Data Source: Open-Meteo API | Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

    return '\n'.join(lines)


# --- Test Response Generator ---
print('=== Natural Language Response Generator Test ===')
print(generate_full_response('London', 'combined'))

## Step 7: Edge Case Testing
**Owner: Nandini**

Tests invalid locations, misspelled cities, and edge scenarios.
Validates fallback logic and graceful error handling.

In [ ]:
def run_edge_case_tests() -> None:
    """
    Runs a suite of edge case tests to validate system robustness.
    """
    print('=== Edge Case Testing Suite ===')
    passed = 0
    failed = 0

    # Test 1: Invalid city name
    print('\nTest 1: Invalid city name')
    result = get_coordinates('XYZInvalidCity123')
    if 'error' in result:
        print(f'PASS — Error handled correctly: {result["error"]}')
        passed += 1
    else:
        print('FAIL — Expected error but got result')
        failed += 1

    # Test 2: Misspelled city
    print('\nTest 2: Misspelled city (Torontoo)')
    result = get_weather('Torontoo')
    if 'error' in result:
        print(f'PASS — Error handled: {result["error"]}')
        passed += 1
    else:
        print(f'INFO — System found a result for misspelled city: {result.get("city")}')
        passed += 1

    # Test 3: Empty string
    print('\nTest 3: Empty city string')
    result = get_coordinates('')
    if 'error' in result:
        print(f'PASS — Empty string handled: {result["error"]}')
        passed += 1
    else:
        print('FAIL — Expected error for empty string')
        failed += 1

    # Test 4: Valid city weather
    print('\nTest 4: Valid city weather (Paris)')
    result = get_weather('Paris')
    if 'error' not in result and 'temperature_celsius' in result:
        print(f'PASS — Weather data retrieved: {result["temperature_celsius"]}C')
        passed += 1
    else:
        print(f'FAIL — {result.get("error", "Unexpected result")}')
        failed += 1

    # Test 5: Trend with empty data
    print('\nTest 5: Trend detection with empty data')
    result = detect_trend([], 'Temperature')
    if result['trend'] == 'stable':
        print('PASS — Empty list returns stable trend')
        passed += 1
    else:
        print('FAIL — Expected stable for empty list')
        failed += 1

    # Test 6: Trend with single value
    print('\nTest 6: Trend detection with single value')
    result = detect_trend([25], 'Temperature')
    if result['trend'] == 'stable':
        print('PASS — Single value returns stable trend')
        passed += 1
    else:
        print('FAIL — Expected stable for single value')
        failed += 1

    # Summary
    print(f'\n=== Test Results: {passed} passed, {failed} failed ===')
    if failed == 0:
        print('All edge case tests passed!')


run_edge_case_tests()

## Step 9: End-to-End Flow Integration & CLI
**Owner: Rohit (Integration) | Kritika (CLI)**

Connects all modules into a single working pipeline.
User query → Intent Classification → Tool Agents → Analytics → Response Generation

In [ ]:
def classify_intent(query: str) -> dict:
    """Simple intent classifier (from Week 7)."""
    query_lower = query.lower()
    weather_keywords = ['weather', 'temperature', 'temp', 'rain', 'wind', 'humid', 'hot', 'cold', 'sunny', 'cloudy', 'snow']
    aqi_keywords = ['aqi', 'air quality', 'pollution', 'pm2.5', 'pm10', 'smog', 'breathe', 'air']
    comparison_keywords = ['compare', 'vs', 'versus', 'difference between']

    has_weather = any(kw in query_lower for kw in weather_keywords)
    has_aqi = any(kw in query_lower for kw in aqi_keywords)
    has_comparison = any(kw in query_lower for kw in comparison_keywords)

    if has_comparison:
        intent = 'comparison'
    elif has_weather and has_aqi:
        intent = 'combined'
    elif has_aqi:
        intent = 'aqi_only'
    elif has_weather:
        intent = 'weather_only'
    else:
        intent = 'combined'

    return {'intent': intent, 'has_weather': has_weather, 'has_aqi': has_aqi}


def extract_city(query: str) -> str:
    """Simple city extractor — gets city from the end of the query."""
    stopwords = ['what', 'is', 'the', 'in', 'for', 'of', 'how', 'give', 'me',
                 'current', 'today', 'right', 'now', 'tell', 'about', 'temperature',
                 'weather', 'aqi', 'air', 'quality', 'pollution', 'report', 'a', 'an']
    words = query.replace('?', '').replace(',', '').split()
    city_words = [w for w in words if w.lower() not in stopwords]
    return ' '.join(city_words[-2:]) if len(city_words) >= 2 else city_words[-1] if city_words else 'Unknown'


def smart_city_pipeline(user_query: str) -> None:
    """
    Full end-to-end pipeline for the Smart City Analytics Platform.

    Args:
        user_query (str): Natural language query from user
    """
    print('\n' + '=' * 60)
    print(f'Query: {user_query}')
    print('=' * 60)

    # Step 1: Classify intent
    intent_result = classify_intent(user_query)
    intent = intent_result['intent']
    print(f'Intent Detected: {intent}')

    # Step 2: Extract city
    city = extract_city(user_query)
    print(f'City Extracted : {city}')
    print('-' * 60)

    # Step 3: Generate full response
    print(generate_full_response(city, intent))


# --- End-to-End Tests ---
print('=== End-to-End Flow Integration Test ===')

test_queries = [
    'What is the weather in Berlin?',
    'How is the air quality in Mumbai?',
    'Give me a full environment report for Toronto'
]

for query in test_queries:
    smart_city_pipeline(query)

## Step 10: Interactive CLI Mode

Run this cell to interact with the full Smart City Platform.
Type any city-based environmental query. Type `quit` to exit.

In [ ]:
print('Smart City Analytics Platform — Interactive Mode')
print('Ask about weather or air quality in any city.')
print('Examples:')
print('  - What is the temperature in Tokyo?')
print('  - How is the air quality in Delhi?')
print('  - Give me a full report for London')
print("Type 'quit' to exit.\n")

while True:
    user_input = input('You: ').strip()
    if user_input.lower() in ['quit', 'exit', 'q']:
        print('Goodbye!')
        break
    if not user_input:
        continue
    smart_city_pipeline(user_input)
    print()